In [60]:
import os
import re
from typing import List, Dict

# Import libraries
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt


# EEG libraries
import mne

In [129]:

def trail_classification(trail_num : int):
    """ 
        function : Classify Trail into {'motor', 'imaginary' and 'baseline'} 
        Input : 
            : trail_num -> trail number
        returns:
            [str] : return classification category
    """
    if trail_num in [4, 6, 8, 10, 12, 14]:
        return "imaginary"
    elif trail_num in [1, 2]:
        return "baseline"
    else:
        return "motor"
    

def event_type_classification(trail_num : int):
    
    if trail_num in [1, 2]:
        return ("rest", "NA", "NA")
    elif trail_num in [3, 4, 7, 8, 11, 12]:
        return ("rest", "left_fist", "right_fist")
    elif trail_num in [5, 6, 9, 10, 13, 14]:
        return ("rest", "both_fist", "bot_feet")


def extract_metadata(data_dir):
    """
    Extract metadata from EDF and EDF.event files.
    
    Parameters:
        data_dir (str): Path to the main data folder containing subject subfolders.
    
    Returns:
        pd.DataFrame: Metadata table with subject, trial, run_type, edf_path, event_path
    """
    
    metadata = []

    # Iterate over subjects
    for subj in sorted(os.listdir(data_dir)):
        subj_path = os.path.join(data_dir, subj)
        if not os.path.isdir(subj_path):
            continue  # skip non-folder items

        # Iterate over files
        for file in sorted(os.listdir(subj_path)):
            if file.endswith(".edf") and not file.endswith(".edf.event"):
                # Extract trial number (last 2 digits before extension)
                match = re.search(r"R(\d{2})\.edf$", file)
                if match:
                    trial_num = int(match.group(1))
                    trail_type = trail_classification(trail_num= trial_num)
                    event_type = event_type_classification(trail_num= trial_num)
                    
                    edf_file = os.path.join(subj_path, file)
                    event_file = edf_file + ".event"

                    metadata.append({
                        "subject": subj,
                        "trial": trial_num,
                        "trail_type": trail_type,
                        "edf_file": edf_file,
                        "event_file": event_file if os.path.exists(event_file) else None,
                        "T0":event_type[0],
                        "T1":event_type[1],
                        "T2":event_type[2],
                    })

    return pd.DataFrame(metadata)

In [131]:
metadata_info = extract_metadata("../data/")
metadata_info

,subject,trial,trail_type,edf_file,event_file,T0,T1,T2
0,S001,1,baseline,../data/S001/S001R01.edf,../data/S001/S001R01.edf.event,rest,NA,NA
1,S001,2,baseline,../data/S001/S001R02.edf,../data/S001/S001R02.edf.event,rest,NA,NA
2,S001,3,motor,../data/S001/S001R03.edf,../data/S001/S001R03.edf.event,rest,left_fist,right_fist
3,S001,4,imaginary,../data/S001/S001R04.edf,../data/S001/S001R04.edf.event,rest,left_fist,right_fist
4,S001,5,motor,../data/S001/S001R05.edf,../data/S001/S001R05.edf.event,rest,both_fist,bot_feet
...,...,...,...,...,...,...,...,...
1521,S109,10,imaginary,../data/S109/S109R10.edf,../data/S109/S109R10.edf.event,rest,both_fist,bot_feet
1522,S109,11,motor,../data/S109/S109R11.edf,../data/S109/S109R11.edf.event,rest,left_fist,right_fist
1523,S109,12,imaginary,../data/S109/S109R12.edf,../data/S109/S109R12.edf.event,rest,left_fist,right_fist
1524,S109,13,motor,../data/S109/S109R13.edf,../data/S109/S109R13.edf.event,rest,both_fist,bot_feet


In [101]:
raw = mne.io.read_raw_edf("../data/S001/S001R04.edf", preload=True)

# Extract annotations
annotations = raw.annotations
print(annotations[:10])

Extracting EDF parameters from /Users/kavisanthoshkumar/Library/CloudStorage/OneDrive-IllinoisInstituteofTechnology/Tensorflow/PhysicoNet/data/S001/S001R04.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 19999  =      0.000 ...   124.994 secs...
<Annotations | 10 segments: T0 (5), T1 (2), T2 (3)>


In [102]:
raw._annotations

<Annotations | 30 segments: T0 (15), T1 (8), T2 (7)>

In [103]:
raw.to_data_frame()

,time,Fc5.,Fc3.,Fc1.,Fcz.,Fc2.,Fc4.,Fc6.,C5..,C3..,...,P8..,Po7.,Po3.,Poz.,Po4.,Po8.,O1..,Oz..,O2..,Iz..
0,0.00000,-5.0,2.0,37.0,39.0,30.0,26.0,-16.0,-14.0,4.0,...,-21.0,-8.0,-35.0,-45.0,-66.0,-39.0,-33.0,-48.0,-39.0,-39.0
1,0.00625,-12.0,-24.0,1.0,-2.0,-15.0,-22.0,-55.0,-36.0,-27.0,...,-50.0,-40.0,-68.0,-65.0,-84.0,-52.0,-21.0,-42.0,-31.0,-34.0
2,0.01250,-77.0,-78.0,-59.0,-65.0,-63.0,-55.0,-67.0,-88.0,-71.0,...,-17.0,-22.0,-50.0,-35.0,-48.0,-18.0,-20.0,-42.0,-29.0,-27.0
3,0.01875,-66.0,-67.0,-50.0,-65.0,-60.0,-55.0,-68.0,-62.0,-53.0,...,-39.0,-60.0,-78.0,-64.0,-68.0,-41.0,-44.0,-62.0,-34.0,-43.0
4,0.02500,-45.0,-55.0,-33.0,-53.0,-54.0,-63.0,-83.0,-52.0,-50.0,...,-44.0,-55.0,-70.0,-54.0,-63.0,-37.0,-60.0,-70.0,-34.0,-45.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19995,124.96875,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
19996,124.97500,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
19997,124.98125,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
19998,124.98750,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [104]:
for annot in raw.annotations:
    print(annot['description'], annot['onset'], annot['duration'])

T0 0.0 4.2
T2 4.2 4.1
T0 8.3 4.2
T1 12.5 4.1
T0 16.6 4.2
T1 20.8 4.1
T0 24.9 4.2
T2 29.1 4.1
T0 33.2 4.2
T2 37.4 4.1
T0 41.5 4.2
T1 45.7 4.1
T0 49.8 4.2
T2 54.0 4.1
T0 58.1 4.2
T1 62.3 4.1
T0 66.4 4.2
T2 70.6 4.1
T0 74.7 4.2
T1 78.9 4.1
T0 83.0 4.2
T1 87.2 4.1
T0 91.3 4.2
T2 95.5 4.1
T0 99.6 4.2
T1 103.8 4.1
T0 107.9 4.2
T2 112.1 4.1
T0 116.2 4.2
T1 120.4 4.1


In [79]:
(4.1 * 160) + 672

1328.0

In [69]:
mne.events_from_annotations(raw)

Used Annotations descriptions: [np.str_('T0'), np.str_('T1'), np.str_('T2')]


(array([[    0,     0,     1],
        [  672,     0,     3],
        [ 1328,     0,     1],
        [ 2000,     0,     2],
        [ 2656,     0,     1],
        [ 3328,     0,     2],
        [ 3984,     0,     1],
        [ 4656,     0,     3],
        [ 5312,     0,     1],
        [ 5984,     0,     3],
        [ 6640,     0,     1],
        [ 7312,     0,     2],
        [ 7968,     0,     1],
        [ 8640,     0,     3],
        [ 9296,     0,     1],
        [ 9968,     0,     2],
        [10624,     0,     1],
        [11296,     0,     3],
        [11952,     0,     1],
        [12624,     0,     2],
        [13280,     0,     1],
        [13952,     0,     2],
        [14608,     0,     1],
        [15280,     0,     3],
        [15936,     0,     1],
        [16608,     0,     2],
        [17264,     0,     1],
        [17936,     0,     3],
        [18592,     0,     1],
        [19264,     0,     2]]),
 {np.str_('T0'): 1, np.str_('T1'): 2, np.str_('T2'): 3})

In [126]:
def getEpochVariableDuration(file_name: str, sfreq:int):

    # Load raw EEG data - present in EDF format
    raw = mne.io.read_raw_edf(file_name, preload=True)


    data = []
    for ann in raw.annotations:
        start = int(ann["onset"] * sfreq) # start time event
        stop = int((ann["onset"] + ann["duration"]) * sfreq) # end time event

        epoch_data = raw.get_data(start = start, 
                                  stop = stop)
        label = ann["description"]

        # Append the epoch_data to data
        data.append((epoch_data, label))

    return data


data = getEpochVariableDuration(file_name= "../data/S001/S001R01.edf", sfreq=160)

Extracting EDF parameters from /Users/kavisanthoshkumar/Library/CloudStorage/OneDrive-IllinoisInstituteofTechnology/Tensorflow/PhysicoNet/data/S001/S001R01.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 9759  =      0.000 ...    60.994 secs...


In [127]:
data

[(array([[-1.6e-05, -5.6e-05, -5.5e-05, ...,  6.0e-06,  7.0e-06,  2.1e-05],
         [-2.9e-05, -5.4e-05, -5.5e-05, ...,  2.9e-05,  3.0e-05,  4.4e-05],
         [ 2.0e-06, -2.7e-05, -2.9e-05, ...,  1.2e-05,  1.1e-05,  2.0e-05],
         ...,
         [-2.1e-05, -1.2e-05,  2.0e-06, ..., -5.0e-05, -5.3e-05, -4.3e-05],
         [-1.1e-05,  1.0e-06,  1.8e-05, ..., -3.6e-05, -4.0e-05, -3.7e-05],
         [ 1.5e-05,  2.1e-05,  3.5e-05, ..., -1.8e-05, -2.2e-05, -8.0e-06]],
        shape=(64, 9632)),
  np.str_('T0'))]

In [122]:
data[1][0].shape

(64, 656)

In [125]:
str(data[0][1])

'T0'